## 1. 路径配置与标签加载

In [ ]:
## 1. 路径配置与标签加载
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib import cm
from matplotlib.patches import Rectangle, Patch
import h5py
import pandas as pd
from collections import Counter
from scipy.stats import entropy

# ============================================================================
# 路径配置 - 修改此处切换被试
# ============================================================================

SUBJECT_ID = "ODP_01_qhlazec"
SPLIT_TYPE = "test"

BASE_RESULT_DIR = Path("/home/jovyan/gpu_space/workspace_jiayi/KAN-git/KAN-Brain-Single-Voxel-Segmentaion/3D_dev/training/downsampling/runs/loso_37fold")

if SPLIT_TYPE == "test":
    fold_candidates = list(BASE_RESULT_DIR.glob(f"*_test_{SUBJECT_ID}"))
    FOLD_DIR = fold_candidates[0] if len(fold_candidates) > 0 else BASE_RESULT_DIR / f"fold_01_test_{SUBJECT_ID}"
else:
    FOLD_DIR = BASE_RESULT_DIR / "fold_01_test_ODP_01_qhlazec"

DATA_ROOT = Path("/home/jovyan/gpu_space/workspace_jiayi/alex_datasets/downsampling/3d")
DOWNSAMPLED_FILE = DATA_ROOT / f"{SUBJECT_ID}_downsampled.npz"
PRED_SOFTMAX_FILE = FOLD_DIR / "pred_3d" / f"{SPLIT_TYPE}_{SUBJECT_ID}_pred_softmax_3d.npz"
GT_3D_FILE = DATA_ROOT / "3d" / f"{SUBJECT_ID}_3d.npz"
LABEL_EXCEL = Path("/home/jovyan/gpu_space/workspace_jiayi/KAN-git/KAN-Brain-Single-Voxel-Segmentaion/3D_dev/training/downsampling/Freesurfer_LUT_alex_labels_jiayi.xlsx")

OUTPUT_DIR = FOLD_DIR / "figs" / "confidence_overlay"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHANNEL_MPRAGE = 341
CHANNEL_QSM = 350

# ============================================================================
# 可视化参数
# ============================================================================

TAU = 0.4
N_CLASSES = 102
SLICE_AXIS = 'axial'
AUTO_SELECT_SLICES = True
MANUAL_SLICES = [30, 50, 70]
CONTOUR_LEVELS = [0.3]
CONTOUR_LINEWIDTH = 1.5

ZOOM_REGIONS = {
    "cortex_white": {"z": None, "x": None, "y": None, "hw": 20},
    "basal_ganglia": {"z": None, "x": None, "y": None, "hw": 20},
    "brainstem": {"z": None, "x": None, "y": None, "hw": 20}
}

print("✓ 路径配置完成")
print(f"  被试: {SUBJECT_ID}, 类型: {SPLIT_TYPE}")
print(f"  Fold: {FOLD_DIR.name}")
print(f"  输出: {OUTPUT_DIR}")

# ============================================================================
# 加载FreeSurfer标签映射
# ============================================================================

print("\n加载FreeSurfer标签映射...")
label_df = pd.read_excel(LABEL_EXCEL)
valid_labels = label_df[label_df['one_hot_loc_alex_label'] != '[]'].copy()
valid_labels['label_idx'] = valid_labels['one_hot_loc_alex_label'].astype(int)

LABEL_NAMES = {0: 'Background'}
LABEL_COLORS_RGB = {0: (0, 0, 0)}

for idx, row in valid_labels.iterrows():
    label_idx = int(row['label_idx'])
    if 1 <= label_idx <= N_CLASSES and label_idx not in LABEL_NAMES:
        LABEL_NAMES[label_idx] = row['tissue_name'].strip("'")
        LABEL_COLORS_RGB[label_idx] = (row['R']/255, row['G']/255, row['B']/255)

colors_list = [LABEL_COLORS_RGB.get(i, (0, 0, 0)) for i in range(N_CLASSES)]
cmap_freesurfer = mcolors.ListedColormap(colors_list)

print(f"✓ 加载了 {len(LABEL_NAMES)} 个标签")

In [ ]:
## 2. 加载数据并计算置信度与不确定性
# print("加载数据...")

# 预测概率
pred_data = np.load(PRED_SOFTMAX_FILE)
pred_softmax = pred_data['pred_softmax_3d']
print(f"  预测概率: {pred_softmax.shape}")

# Ground Truth
gt_data = np.load(GT_3D_FILE)
gt_proba = gt_data['proba_labels']
region_mask = gt_data['region_mask_lr']
print(f"  Ground Truth: {gt_proba.shape}")

# MPRAGE底图
if DOWNSAMPLED_FILE.exists():
    downsampled_data = np.load(DOWNSAMPLED_FILE)
    data_lr = downsampled_data['data_lr']
    anatomy_img = data_lr[..., CHANNEL_MPRAGE]
    anatomy_name = "MPRAGE"
    print(f"  MPRAGE底图: {anatomy_img.shape}")
else:
    anatomy_img = np.zeros(pred_softmax.shape[:3])
    anatomy_name = "No Anatomy"

# ============================================================================
# 计算置信度
# ============================================================================

print("\n计算置信度与不确定性...")

top2_indices = np.argsort(pred_softmax, axis=-1)[..., -2:]
top1_class = top2_indices[..., 1]
top2_class = top2_indices[..., 0]

top2_probs = np.take_along_axis(pred_softmax, top2_indices, axis=-1)
p1 = top2_probs[..., 1]
p2 = top2_probs[..., 0]

# 边际差 (margin)
margin = p1 - p2  # (Z, X, Y)
alpha_map = np.clip(margin / TAU, 0, 1)
alpha_map[region_mask == 0] = 0

# ============================================================================
# 计算熵图（不确定性）
# ============================================================================

# H(p) = -Σ_k p_k log(p_k)
# 使用scipy.stats.entropy，axis=-1计算每个体素的熵
epsilon = 1e-10  # 避免log(0)
pred_softmax_safe = np.clip(pred_softmax, epsilon, 1.0)

# entropy使用自然对数，转换为bits需要除以log(2)
entropy_map = entropy(pred_softmax_safe.T, axis=0).T  # scipy要求axis=0，所以转置
entropy_map = entropy_map / np.log(2)  # 转为bits

# 应用ROI掩码
entropy_map_masked = entropy_map.copy()
entropy_map_masked[region_mask == 0] = 0

# 统计
entropy_roi = entropy_map[region_mask > 0]
margin_roi = margin[region_mask > 0]

print(f"  边际差 (Margin, ROI内):")
print(f"    Mean={margin_roi.mean():.4f}, Median={np.median(margin_roi):.4f}")
print(f"    Range=[{margin_roi.min():.4f}, {margin_roi.max():.4f}]")

print(f"  熵 (Entropy, ROI内):")
print(f"    Mean={entropy_roi.mean():.4f} bits, Median={np.median(entropy_roi):.4f} bits")
print(f"    Range=[{entropy_roi.min():.4f}, {entropy_roi.max():.4f}] bits")
print(f"    Max possible: {np.log2(N_CLASSES):.2f} bits (uniform distribution)")

# ============================================================================
# 自动选择切片
# ============================================================================

if AUTO_SELECT_SLICES:
    print("\n自动选择切片...")
    axis_size = pred_softmax.shape[0]
    slice_scores = []
    
    for z in range(axis_size):
        mask_slice = region_mask[z] > 0
        if mask_slice.sum() > 100:
            # 使用熵的标准差作为信息量度量
            entropy_std = entropy_map[z][mask_slice].std()
            mask_ratio = mask_slice.sum() / mask_slice.size
            slice_scores.append((z, entropy_std * mask_ratio))
    
    slice_scores.sort(key=lambda x: x[1], reverse=True)
    selected_slices = []
    for z, score in slice_scores:
        if len(selected_slices) == 0 or all(abs(z - s) >= 10 for s in selected_slices):
            selected_slices.append(z)
        if len(selected_slices) >= 3:
            break
    
    MANUAL_SLICES = sorted(selected_slices)
    print(f"  选择的切片: {MANUAL_SLICES}")

print("\n✓ 数据准备完成")

---

# Figure 4: Boundary vs Interior Analysis (post-T)

**完整流程**：边界检测 → 距离变换 → 指标计算 → 统计分析 → 可视化

一键运行下面的 cell 即可生成完整的 Figure 4。

In [ ]:
## 保存 Figure 4 到文件

# 检查是否存在 fig 和 results
if 'fig' not in locals() or 'results' not in locals():
    print("⚠ 请先运行上面的 Figure 4 生成 cell！")
else:
    # 输出目录（使用之前定义的 OUTPUT_DIR）
    save_dir = OUTPUT_DIR if 'OUTPUT_DIR' in globals() else Path("./output")
    save_dir.mkdir(parents=True, exist_ok=True)
    
    # 保存图像
    png_file = save_dir / 'Fig4_boundary_analysis_postT.png'
    pdf_file = save_dir / 'Fig4_boundary_analysis_postT.pdf'
    
    fig.savefig(png_file, dpi=600, bbox_inches='tight', facecolor='white')
    fig.savefig(pdf_file, bbox_inches='tight', facecolor='white')
    
    print(f"✓ 已保存 PNG: {png_file}")
    print(f"✓ 已保存 PDF: {pdf_file}")
    
    # 保存统计数据到 CSV
    import pandas as pd
    
    n_folds = len(results['edge_coverage'])
    stats_df = pd.DataFrame({
        'fold_idx': range(n_folds),
        'edge_coverage': results['edge_coverage'],
        'interior_coverage': results['interior_coverage'],
        'edge_ece': results['edge_ece'],
        'interior_ece': results['interior_ece']
    })
    
    csv_file = save_dir / 'Fig4_boundary_stats.csv'
    stats_df.to_csv(csv_file, index=False)
    
    print(f"✓ 已保存统计数据: {csv_file}")
    print(f"\n📊 统计摘要:")
    print(f"  Edge Coverage:    {results['edge_coverage'].mean():.4f} ± {results['edge_coverage'].std():.4f}")
    print(f"  Interior Coverage: {results['interior_coverage'].mean():.4f} ± {results['interior_coverage'].std():.4f}")
    print(f"  Edge ECE:         {results['edge_ece'].mean():.4f} ± {results['edge_ece'].std():.4f}")
    print(f"  Interior ECE:     {results['interior_ece'].mean():.4f} ± {results['interior_ece'].std():.4f}")

## 可选：保存 Figure 4 图像

运行下面的 cell 保存高分辨率图像（600 dpi PNG + PDF）和统计数据（CSV）。

In [ ]:
## Figure 4: Edge vs Interior - Production Code (Multi-Fold Version)
# ============================================================================
# Process all 37 folds with violin plots
# ============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import distance_transform_edt
from scipy.stats import wilcoxon
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# Core Functions
# ============================================================================

def compute_boundary_mask(y_int):
    """Compute boundary mask using 6-connectivity (3D)."""
    foreground = y_int > 0
    boundary = np.zeros_like(y_int, dtype=bool)
    
    if y_int.ndim == 3:
        for dz, dy, dx in [(-1,0,0), (1,0,0), (0,-1,0), (0,1,0), (0,0,-1), (0,0,1)]:
            shifted = np.roll(y_int, shift=(dz, dy, dx), axis=(0, 1, 2))
            boundary |= (y_int != shifted) & foreground
    else:
        raise ValueError(f"y_int must be 3D, got shape {y_int.shape}")
    
    return boundary


def distance_to_boundary_mm(y_int, voxel_spacing_mm):
    """Compute Euclidean distance to nearest boundary in mm."""
    boundary = compute_boundary_mask(y_int)
    dist_mm = distance_transform_edt(~boundary, sampling=voxel_spacing_mm)
    return dist_mm


def make_edge_interior_masks(y_int, voxel_spacing_mm, d_edge_mm=2.0, d_interior_mm=3.0):
    """Create Edge and Interior masks based on distance to boundary."""
    dist_mm = distance_to_boundary_mm(y_int, voxel_spacing_mm)
    foreground = y_int > 0
    
    mask_edge = (dist_mm <= d_edge_mm) & foreground
    mask_interior = (dist_mm >= d_interior_mm) & foreground
    
    return mask_edge.ravel(), mask_interior.ravel()


def soft_ece_postT(P_postT, Q_soft, mask, n_bins=15):
    """Compute soft Expected Calibration Error (post-T)."""
    P = P_postT[mask]
    Q = Q_soft[mask]
    
    if P.shape[0] == 0:
        return np.nan
    
    # Renormalize if needed
    prob_sums = P.sum(axis=1)
    if np.abs(prob_sums - 1.0).max() > 1e-3:
        P = P / (prob_sums[:, np.newaxis] + 1e-10)
    
    P = np.clip(P, 0, 1)
    
    conf = P.max(axis=1)
    soft_acc = (P * Q).sum(axis=1)
    
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    
    for i in range(n_bins):
        if i == n_bins - 1:
            bin_mask = (conf >= bin_edges[i]) & (conf <= bin_edges[i + 1])
        else:
            bin_mask = (conf >= bin_edges[i]) & (conf < bin_edges[i + 1])
        
        n_b = bin_mask.sum()
        if n_b > 0:
            mean_conf = conf[bin_mask].mean()
            mean_soft_acc = soft_acc[bin_mask].mean()
            ece += np.abs(mean_conf - mean_soft_acc) * (n_b / P.shape[0])
    
    return ece


def top3_union_postT(P_postT, y_int, mask):
    """Compute Top-3 union coverage (post-T)."""
    P = P_postT[mask]
    y = y_int[mask]
    
    if P.shape[0] == 0:
        return np.nan
    
    # Renormalize if needed
    prob_sums = P.sum(axis=1)
    if np.abs(prob_sums - 1.0).max() > 1e-3:
        P = P / (prob_sums[:, np.newaxis] + 1e-10)
    
    P = np.clip(P, 0, 1)
    
    top3_indices = np.argsort(P, axis=1)[:, -3:]
    matches = np.any(top3_indices == y[:, np.newaxis], axis=1)
    coverage = matches.mean()
    
    return coverage


def top1_accuracy(P_postT, y_int, mask):
    """Compute Top-1 accuracy (NOT post-T - argmax unchanged)."""
    P = P_postT[mask]
    y = y_int[mask]
    
    if P.shape[0] == 0:
        return np.nan
    
    pred = P.argmax(axis=1)
    accuracy = (pred == y).mean()
    
    return accuracy


# ============================================================================
# Main Analysis - Multi-Fold Processing
# ============================================================================

print("=" * 80)
print("Figure 4: Edge vs Interior Analysis (Multi-Fold Production Version)")
print("=" * 80)

# Configuration
VOXEL_SPACING = (3.0, 1.8, 1.8)  # (Z, Y, X) mm
EDGE_THRESHOLD = 2.0  # mm
INTERIOR_THRESHOLD = 3.0  # mm
N_BINS_ECE = 15

BASE_RESULT_DIR = Path("/home/jovyan/gpu_space/workspace_jiayi/KAN-git/KAN-Brain-Single-Voxel-Segmentaion/3D_dev/training/downsampling/runs/loso_37fold")
DATA_ROOT = Path("/home/jovyan/gpu_space/workspace_jiayi/alex_datasets/downsampling/3d")

fold_dirs = sorted(list(BASE_RESULT_DIR.glob("fold_*_test_*")))

print(f"\nFound {len(fold_dirs)} folds")
print(f"Voxel spacing: {VOXEL_SPACING} mm (Z, Y, X)")
print(f"Edge threshold: ≤ {EDGE_THRESHOLD} mm")
print(f"Interior threshold: ≥ {INTERIOR_THRESHOLD} mm\n")

# Collect results
results = {
    'edge_top3': [],
    'interior_top3': [],
    'edge_ece': [],
    'interior_ece': [],
    'edge_acc': [],
    'interior_acc': []
}

# Process each fold
for fold_idx, fold_dir in enumerate(fold_dirs):
    fold_name = fold_dir.name
    print(f"[{fold_idx + 1}/{len(fold_dirs)}] {fold_name}")
    
    try:
        # Parse subject ID
        parts = fold_name.split('_test_')
        if len(parts) != 2:
            print(f"  ⚠ Skip: cannot parse subject ID\n")
            continue
        subject_id = parts[1]
        
        # File paths
        pred_file = fold_dir / "pred_3d" / f"test_{subject_id}_pred_softmax_3d.npz"
        gt_file = DATA_ROOT / "3d" / f"{subject_id}_3d.npz"
        
        if not pred_file.exists() or not gt_file.exists():
            print(f"  ⚠ Skip: files not found\n")
            continue
        
        # Load data
        pred_data = np.load(pred_file)
        gt_data = np.load(gt_file)
        
        P_postT = pred_data['pred_softmax_3d']  # [Z, Y, X, C]
        Q_soft = gt_data['proba_labels']  # [Z, Y, X, C]
        region_mask = gt_data['region_mask_lr']  # [Z, Y, X]
        
        # Create hard labels from soft reference
        y_int = np.argmax(Q_soft, axis=-1).astype(np.int32)  # [Z, Y, X]
        y_int[region_mask == 0] = -1  # Mark background
        
        # Flatten
        Z, Y, X, C = P_postT.shape
        P_postT_flat = P_postT.reshape(-1, C)
        Q_soft_flat = Q_soft.reshape(-1, C)
        y_int_flat = y_int.ravel()
        
        # Filter valid labels
        valid = (y_int_flat >= 0) & (y_int_flat < C)
        P_postT_flat = P_postT_flat[valid]
        Q_soft_flat = Q_soft_flat[valid]
        y_int_flat = y_int_flat[valid]
        
        # Compute Edge/Interior masks
        mask_edge_spatial, mask_interior_spatial = make_edge_interior_masks(
            y_int, VOXEL_SPACING, d_edge_mm=EDGE_THRESHOLD, d_interior_mm=INTERIOR_THRESHOLD
        )
        
        mask_edge = mask_edge_spatial[valid]
        mask_interior = mask_interior_spatial[valid]
        
        n_edge = mask_edge.sum()
        n_interior = mask_interior.sum()
        
        if n_edge < 100 or n_interior < 100:
            print(f"  ⚠ Skip: Edge={n_edge}, Interior={n_interior} (< 100 voxels)\n")
            continue
        
        # Compute metrics
        top3_edge = top3_union_postT(P_postT_flat, y_int_flat, mask_edge)
        top3_int = top3_union_postT(P_postT_flat, y_int_flat, mask_interior)
        
        ece_edge = soft_ece_postT(P_postT_flat, Q_soft_flat, mask_edge, n_bins=N_BINS_ECE)
        ece_int = soft_ece_postT(P_postT_flat, Q_soft_flat, mask_interior, n_bins=N_BINS_ECE)
        
        acc_edge = top1_accuracy(P_postT_flat, y_int_flat, mask_edge)
        acc_int = top1_accuracy(P_postT_flat, y_int_flat, mask_interior)
        
        # Store results
        results['edge_top3'].append(top3_edge)
        results['interior_top3'].append(top3_int)
        results['edge_ece'].append(ece_edge)
        results['interior_ece'].append(ece_int)
        results['edge_acc'].append(acc_edge)
        results['interior_acc'].append(acc_int)
        
        print(f"  ✓ Top3: E={top3_edge:.3f}, I={top3_int:.3f} | "
              f"ECE: E={ece_edge:.4f}, I={ece_int:.4f} | "
              f"Acc: E={acc_edge:.3f}, I={acc_int:.3f}\n")
        
    except Exception as e:
        print(f"  ✗ Error: {e}\n")
        continue

# Convert to arrays
for key in results:
    results[key] = np.array(results[key])

n_valid = len(results['edge_top3'])
print("=" * 80)
print(f"✓ Completed! Processed {n_valid} valid folds")
print("=" * 80)

if n_valid == 0:
    print("\n⚠ No valid data, cannot plot")
else:
    # ========================================================================
    # Statistical Summary
    # ========================================================================
    
    print("\n" + "=" * 80)
    print("Statistical Summary (Median [Q25, Q75])")
    print("=" * 80)
    
    def print_stats(data, name):
        med, q25, q75 = np.median(data), np.percentile(data, 25), np.percentile(data, 75)
        print(f"{name:35s}: {med:.4f} [{q25:.4f}, {q75:.4f}]")
        return med, q25, q75
    
    print("\n【Top-3 union (post-T)】")
    edge_top3_med, _, _ = print_stats(results['edge_top3'], "  Edge (≤2mm)")
    int_top3_med, _, _ = print_stats(results['interior_top3'], "  Interior (≥3mm)")
    
    print("\n【Soft-ECE (post-T, soft reference)】")
    edge_ece_med, _, _ = print_stats(results['edge_ece'], "  Edge (≤2mm)")
    int_ece_med, _, _ = print_stats(results['interior_ece'], "  Interior (≥3mm)")
    
    print("\n【Top-1 accuracy (not post-T)】")
    edge_acc_med, _, _ = print_stats(results['edge_acc'], "  Edge (≤2mm)")
    int_acc_med, _, _ = print_stats(results['interior_acc'], "  Interior (≥3mm)")
    
    # Wilcoxon tests
    print("\n" + "=" * 80)
    print("Wilcoxon Signed-Rank Test (Edge vs Interior)")
    print("=" * 80)
    
    try:
        stat_top3, p_top3 = wilcoxon(results['edge_top3'], results['interior_top3'])
        stat_ece, p_ece = wilcoxon(results['edge_ece'], results['interior_ece'])
        stat_acc, p_acc = wilcoxon(results['edge_acc'], results['interior_acc'])
        print(f"Top-3 union:     W={stat_top3:.2f}, p={p_top3:.4e}")
        print(f"Soft-ECE:        W={stat_ece:.2f}, p={p_ece:.4e}")
        print(f"Top-1 accuracy:  W={stat_acc:.2f}, p={p_acc:.4e}")
    except Exception as e:
        print(f"⚠ Wilcoxon test failed: {e}")
    
    # ========================================================================
    # Plotting - Violin plots
    # ========================================================================
    
    print("\n" + "=" * 80)
    print("Generating Figure 4...")
    print("=" * 80)
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.patch.set_facecolor('white')
    
    labels = ['Edge\n(≤2mm)', 'Interior\n(≥3mm)']
    colors = ['#FF6B6B', '#4ECDC4']
    
    # (a) Top-3 union (post-T)
    data_top3 = [results['edge_top3'], results['interior_top3']]
    parts_a = axes[0].violinplot(data_top3, positions=[1, 2], showmeans=False, 
                                  showmedians=True, widths=0.6)
    for i, pc in enumerate(parts_a['bodies']):
        pc.set_facecolor(colors[i])
        pc.set_alpha(0.7)
        pc.set_edgecolor('black')
        pc.set_linewidth(1.5)
    parts_a['cmedians'].set_color('black')
    parts_a['cmedians'].set_linewidth(2.5)
    
    axes[0].boxplot(data_top3, positions=[1, 2], widths=0.3, patch_artist=True, 
                   showfliers=False,
                   boxprops=dict(facecolor='none', edgecolor='black', linewidth=1.5),
                   whiskerprops=dict(color='black', linewidth=1.5),
                   capprops=dict(color='black', linewidth=1.5),
                   medianprops=dict(color='darkred', linewidth=2.5))
    
    axes[0].text(1, edge_top3_med + 0.02, f'{edge_top3_med:.3f}', 
                ha='center', va='bottom', fontsize=9, fontweight='bold')
    axes[0].text(2, int_top3_med + 0.02, f'{int_top3_med:.3f}', 
                ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    axes[0].set_ylabel('Coverage', fontsize=11, fontweight='bold')
    axes[0].set_xticks([1, 2])
    axes[0].set_xticklabels(labels, fontsize=10)
    axes[0].set_ylim([0, 1])
    axes[0].grid(axis='y', alpha=0.3, linestyle='--')
    axes[0].set_title('(a) Top-3 union (post-T)', fontsize=12, fontweight='bold', pad=10)
    
    # (b) Soft-ECE (post-T)
    data_ece = [results['edge_ece'], results['interior_ece']]
    parts_b = axes[1].violinplot(data_ece, positions=[1, 2], showmeans=False, 
                                  showmedians=True, widths=0.6)
    for i, pc in enumerate(parts_b['bodies']):
        pc.set_facecolor(colors[i])
        pc.set_alpha(0.7)
        pc.set_edgecolor('black')
        pc.set_linewidth(1.5)
    parts_b['cmedians'].set_color('black')
    parts_b['cmedians'].set_linewidth(2.5)
    
    axes[1].boxplot(data_ece, positions=[1, 2], widths=0.3, patch_artist=True, 
                   showfliers=False,
                   boxprops=dict(facecolor='none', edgecolor='black', linewidth=1.5),
                   whiskerprops=dict(color='black', linewidth=1.5),
                   capprops=dict(color='black', linewidth=1.5),
                   medianprops=dict(color='darkred', linewidth=2.5))
    
    axes[1].text(1, edge_ece_med + 0.002, f'{edge_ece_med:.4f}', 
                ha='center', va='bottom', fontsize=9, fontweight='bold')
    axes[1].text(2, int_ece_med + 0.002, f'{int_ece_med:.4f}', 
                ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    axes[1].set_ylabel('ECE', fontsize=11, fontweight='bold')
    axes[1].set_xticks([1, 2])
    axes[1].set_xticklabels(labels, fontsize=10)
    axes[1].set_ylim([0, max(data_ece[0].max(), data_ece[1].max()) * 1.15])
    axes[1].grid(axis='y', alpha=0.3, linestyle='--')
    axes[1].set_title('(b) soft-ECE (post-T, soft reference)', fontsize=12, fontweight='bold', pad=10)
    
    # (c) Top-1 accuracy (NOT post-T)
    data_acc = [results['edge_acc'], results['interior_acc']]
    parts_c = axes[2].violinplot(data_acc, positions=[1, 2], showmeans=False, 
                                  showmedians=True, widths=0.6)
    for i, pc in enumerate(parts_c['bodies']):
        pc.set_facecolor(colors[i])
        pc.set_alpha(0.7)
        pc.set_edgecolor('black')
        pc.set_linewidth(1.5)
    parts_c['cmedians'].set_color('black')
    parts_c['cmedians'].set_linewidth(2.5)
    
    axes[2].boxplot(data_acc, positions=[1, 2], widths=0.3, patch_artist=True, 
                   showfliers=False,
                   boxprops=dict(facecolor='none', edgecolor='black', linewidth=1.5),
                   whiskerprops=dict(color='black', linewidth=1.5),
                   capprops=dict(color='black', linewidth=1.5),
                   medianprops=dict(color='darkred', linewidth=2.5))
    
    axes[2].text(1, edge_acc_med + 0.02, f'{edge_acc_med:.3f}', 
                ha='center', va='bottom', fontsize=9, fontweight='bold')
    axes[2].text(2, int_acc_med + 0.02, f'{int_acc_med:.3f}', 
                ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    axes[2].set_ylabel('Accuracy', fontsize=11, fontweight='bold')
    axes[2].set_xticks([1, 2])
    axes[2].set_xticklabels(labels, fontsize=10)
    axes[2].set_ylim([0, 1])
    axes[2].grid(axis='y', alpha=0.3, linestyle='--')
    axes[2].set_title('(c) Top-1 accuracy (not post-T)', fontsize=12, fontweight='bold', pad=10)
    
    # Suptitle
    fig.suptitle(f'Figure 4 — Interface vs interior (post-T calibration & correctness, n={n_valid} folds)',
                 fontsize=14, fontweight='bold', y=0.98)
    
    # Footnote
    footnote = ("Edge: dist≤2 mm; Interior: dist≥3 mm; confidence bins=15; "
                "temperature-scaled metrics reported post-T only. "
                "Violin plots show distribution with overlaid box plots (median as thick line).")
    fig.text(0.5, 0.02, footnote, ha='center', fontsize=8, style='italic', color='gray')
    
    plt.tight_layout(rect=[0, 0.04, 1, 0.96])
    
    # ========================================================================
    # Save outputs
    # ========================================================================
    
    output_dir = Path(OUTPUT_DIR) if 'OUTPUT_DIR' in globals() else Path("./output")
    output_dir.mkdir(parents=True, exist_ok=True)
    
    png_file = output_dir / "figure4_edge_interior.png"
    csv_file = output_dir / "figure4_edge_interior_metrics.csv"
    
    fig.savefig(png_file, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"\n✓ Saved PNG: {png_file}")
    
    # Create DataFrame with all folds
    df_all = pd.DataFrame({
        'fold_idx': range(n_valid),
        'edge_top3_union_postT': results['edge_top3'],
        'interior_top3_union_postT': results['interior_top3'],
        'edge_soft_ece_postT': results['edge_ece'],
        'interior_soft_ece_postT': results['interior_ece'],
        'edge_top1_accuracy': results['edge_acc'],
        'interior_top1_accuracy': results['interior_acc']
    })
    
    df_all.to_csv(csv_file, index=False, float_format='%.6f')
    print(f"✓ Saved CSV: {csv_file}")
    
    plt.show()
    
    # ========================================================================
    # Caption
    # ========================================================================
    
    caption = (
        "Fig. 4. Interface vs interior analysis (post-T). At cortical–WM interfaces (≤2 mm), "
        "Top-3 union coverage remains high while soft-ECE is modestly elevated compared to "
        "interiors (≥3 mm), consistent with partial-volume effects and explainable Top-K behavior. "
        "Top-1 accuracy (not post-T) shows similar patterns. Each fold represents one held-out "
        "subject in LOSO cross-validation. Violin plots show distribution with overlaid box plots "
        "(median as thick red line). Evaluation uses anti-aliased soft references from the "
        "downsampling pipeline. Temperature-scaled metrics reported post-T only."
    )
    
    print("\n" + "=" * 80)
    print("Caption")
    print("=" * 80)
    print(caption)
    
    print("\n✓ Figure 4 generation completed!")

In [ ]:
## Figure 4 综合版 - 2×3 布局合并图
# ============================================================================
# 第一行：Edge vs Interior (violin plots)
# 第二行：Edge vs Interior (bar plots) - Top-1 & Top-3 only
# ============================================================================

# 检查数据是否存在
if 'results' not in locals() or len(results.get('edge_top3', [])) == 0:
    print("⚠ 请先运行上面的 Multi-Fold Production Code！")
else:
    print("=" * 80)
    print("生成 Figure 4 综合版 (2×3 布局)...")
    print("=" * 80)
    
    n_valid = len(results['edge_top3'])
    
    # 计算中位数
    med_acc_edge = np.median(results['edge_acc'])
    med_acc_int = np.median(results['interior_acc'])
    med_top3_edge = np.median(results['edge_top3'])
    med_top3_int = np.median(results['interior_top3'])
    med_ece_edge = np.median(results['edge_ece'])
    med_ece_int = np.median(results['interior_ece'])
    
    print(f"  Valid folds: {n_valid}")
    
    # ========================================================================
    # 创建 2×3 图形
    # ========================================================================
    
    fig = plt.figure(figsize=(18, 10))
    fig.patch.set_facecolor('white')
    
    # 创建 GridSpec 布局
    import matplotlib.gridspec as gridspec
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.3,
                          left=0.08, right=0.95, top=0.93, bottom=0.08)
    
    colors = ['#FF6B6B', '#4ECDC4']
    labels = ['Edge\n(≤2mm)', 'Interior\n(≥3mm)']
    
    # ========================================================================
    # 第一行：Edge vs Interior violin plots
    # ========================================================================
    
    # (a) Top-1 accuracy
    ax1 = fig.add_subplot(gs[0, 0])
    data_acc = [results['edge_acc'], results['interior_acc']]
    parts = ax1.violinplot(data_acc, positions=[1, 2], showmeans=False, 
                           showmedians=True, widths=0.6)
    for i, pc in enumerate(parts['bodies']):
        pc.set_facecolor(colors[i])
        pc.set_alpha(0.7)
        pc.set_edgecolor('black')
        pc.set_linewidth(1.5)
    parts['cmedians'].set_color('black')
    parts['cmedians'].set_linewidth(2.5)
    
    ax1.boxplot(data_acc, positions=[1, 2], widths=0.3, patch_artist=True, 
               showfliers=False,
               boxprops=dict(facecolor='none', edgecolor='black', linewidth=1.5),
               whiskerprops=dict(color='black', linewidth=1.5),
               capprops=dict(color='black', linewidth=1.5),
               medianprops=dict(color='darkred', linewidth=2.5))
    
    ax1.text(1, med_acc_edge + 0.02, f'{med_acc_edge:.3f}', 
            ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax1.text(2, med_acc_int + 0.02, f'{med_acc_int:.3f}', 
            ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    ax1.set_ylabel('Accuracy', fontsize=11, fontweight='bold')
    ax1.set_xticks([1, 2])
    ax1.set_xticklabels(labels, fontsize=10)
    ax1.set_ylim([0, 1])
    ax1.grid(axis='y', alpha=0.3, linestyle='--')
    ax1.set_title('(a) Top-1 accuracy (not post-T)\nEdge vs Interior', 
                 fontsize=11, fontweight='bold', pad=8)
    
    # (b) Top-3 union
    ax2 = fig.add_subplot(gs[0, 1])
    data_top3 = [results['edge_top3'], results['interior_top3']]
    parts = ax2.violinplot(data_top3, positions=[1, 2], showmeans=False, 
                           showmedians=True, widths=0.6)
    for i, pc in enumerate(parts['bodies']):
        pc.set_facecolor(colors[i])
        pc.set_alpha(0.7)
        pc.set_edgecolor('black')
        pc.set_linewidth(1.5)
    parts['cmedians'].set_color('black')
    parts['cmedians'].set_linewidth(2.5)
    
    ax2.boxplot(data_top3, positions=[1, 2], widths=0.3, patch_artist=True, 
               showfliers=False,
               boxprops=dict(facecolor='none', edgecolor='black', linewidth=1.5),
               whiskerprops=dict(color='black', linewidth=1.5),
               capprops=dict(color='black', linewidth=1.5),
               medianprops=dict(color='darkred', linewidth=2.5))
    
    ax2.text(1, med_top3_edge + 0.02, f'{med_top3_edge:.3f}', 
            ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax2.text(2, med_top3_int + 0.02, f'{med_top3_int:.3f}', 
            ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    ax2.set_ylabel('Coverage', fontsize=11, fontweight='bold')
    ax2.set_xticks([1, 2])
    ax2.set_xticklabels(labels, fontsize=10)
    ax2.set_ylim([0, 1])
    ax2.grid(axis='y', alpha=0.3, linestyle='--')
    ax2.set_title('(b) Top-3 union (post-T)\nEdge vs Interior', 
                 fontsize=11, fontweight='bold', pad=8)
    
    # (c) Soft-ECE
    ax3 = fig.add_subplot(gs[0, 2])
    data_ece = [results['edge_ece'], results['interior_ece']]
    parts = ax3.violinplot(data_ece, positions=[1, 2], showmeans=False, 
                           showmedians=True, widths=0.6)
    for i, pc in enumerate(parts['bodies']):
        pc.set_facecolor(colors[i])
        pc.set_alpha(0.7)
        pc.set_edgecolor('black')
        pc.set_linewidth(1.5)
    parts['cmedians'].set_color('black')
    parts['cmedians'].set_linewidth(2.5)
    
    ax3.boxplot(data_ece, positions=[1, 2], widths=0.3, patch_artist=True, 
               showfliers=False,
               boxprops=dict(facecolor='none', edgecolor='black', linewidth=1.5),
               whiskerprops=dict(color='black', linewidth=1.5),
               capprops=dict(color='black', linewidth=1.5),
               medianprops=dict(color='darkred', linewidth=2.5))
    
    ax3.text(1, med_ece_edge + 0.002, f'{med_ece_edge:.4f}', 
            ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax3.text(2, med_ece_int + 0.002, f'{med_ece_int:.4f}', 
            ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    ax3.set_ylabel('ECE', fontsize=11, fontweight='bold')
    ax3.set_xticks([1, 2])
    ax3.set_xticklabels(labels, fontsize=10)
    ax3.set_ylim([0, max(data_ece[0].max(), data_ece[1].max()) * 1.15])
    ax3.grid(axis='y', alpha=0.3, linestyle='--')
    ax3.set_title('(c) Soft-ECE (post-T)\nEdge vs Interior', 
                 fontsize=11, fontweight='bold', pad=8)
    
    # ========================================================================
    # 第二行：Edge vs Interior bar plots (Top-1 & Top-3 only)
    # ========================================================================
    
    # (d) Top-1 accuracy - 条形图
    ax4 = fig.add_subplot(gs[1, 0])
    x_pos = [0, 1]
    values_acc = [med_acc_edge, med_acc_int]
    bars = ax4.bar(x_pos, values_acc, width=0.6, color=colors,
                   edgecolor='black', linewidth=2)
    
    for i, (bar, val) in enumerate(zip(bars, values_acc)):
        ax4.text(bar.get_x() + bar.get_width()/2, val + 0.02,
                f'{val:.3f}', ha='center', va='bottom', 
                fontsize=10, fontweight='bold')
    
    ax4.set_ylabel('Accuracy', fontsize=11, fontweight='bold')
    ax4.set_xticks(x_pos)
    ax4.set_xticklabels(labels, fontsize=10)
    ax4.set_ylim([0, 1])
    ax4.grid(axis='y', alpha=0.3, linestyle='--')
    ax4.set_title(f'(d) Top-1 accuracy (not post-T)\nMedian across {n_valid} folds',
                 fontsize=11, fontweight='bold', pad=8)
    
    # (e) Top-3 union - 条形图
    ax5 = fig.add_subplot(gs[1, 1])
    values_top3 = [med_top3_edge, med_top3_int]
    bars = ax5.bar(x_pos, values_top3, width=0.6, color=colors,
                   edgecolor='black', linewidth=2)
    
    for i, (bar, val) in enumerate(zip(bars, values_top3)):
        ax5.text(bar.get_x() + bar.get_width()/2, val + 0.02,
                f'{val:.3f}', ha='center', va='bottom',
                fontsize=10, fontweight='bold')
    
    ax5.set_ylabel('Coverage', fontsize=11, fontweight='bold')
    ax5.set_xticks(x_pos)
    ax5.set_xticklabels(labels, fontsize=10)
    ax5.set_ylim([0, 1])
    ax5.grid(axis='y', alpha=0.3, linestyle='--')
    ax5.set_title(f'(e) Top-3 union (post-T)\nMedian across {n_valid} folds',
                 fontsize=11, fontweight='bold', pad=8)
    
    # (f) Sample statistics - Text info
    ax6 = fig.add_subplot(gs[1, 2])
    ax6.axis('off')
    
    info_text = f"""
    Sample Statistics
    {'='*30}
    
    Valid folds: {n_valid}
    
    Edge vs Interior:
      • Edge (≤2mm): {n_valid} folds
      • Interior (≥3mm): {n_valid} folds
    
    Configuration:
      • Voxel spacing: (3.0, 1.8, 1.8) mm
      • Edge threshold: ≤ 2.0 mm
      • Interior threshold: ≥ 3.0 mm
      • ECE bins: 15
    
    Metrics (Median):
      • Top-1 acc (not post-T):
        Edge: {med_acc_edge:.3f}
        Interior: {med_acc_int:.3f}
      
      • Top-3 union (post-T):
        Edge: {med_top3_edge:.3f}
        Interior: {med_top3_int:.3f}
      
      • Soft-ECE (post-T):
        Edge: {med_ece_edge:.4f}
        Interior: {med_ece_int:.4f}
    """
    
    ax6.text(0.05, 0.95, info_text, transform=ax6.transAxes,
            fontsize=8.5, verticalalignment='top', family='monospace',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
    
    # ========================================================================
    # 总标题和脚注
    # ========================================================================
    
    fig.suptitle(f'Figure 4 — Edge vs Interior Analysis (n={n_valid} folds)',
                 fontsize=15, fontweight='bold', y=0.97)
    
    footnote = (
        "Top row: Violin+box plots showing distribution across folds. "
        "Bottom row: Bar plots showing median values. "
        "Edge (≤2mm from boundaries) vs Interior (≥3mm). "
        "Temperature-scaled metrics reported post-T only."
    )
    fig.text(0.5, 0.02, footnote, ha='center', fontsize=8, style='italic', color='gray', wrap=True)
    
    # ========================================================================
    # 保存
    # ========================================================================
    
    output_dir = Path(OUTPUT_DIR) if 'OUTPUT_DIR' in globals() else Path("./output")
    output_dir.mkdir(parents=True, exist_ok=True)
    
    png_file = output_dir / "figure4_comprehensive_2x3.png"
    pdf_file = output_dir / "figure4_comprehensive_2x3.pdf"
    
    fig.savefig(png_file, dpi=300, bbox_inches='tight', facecolor='white')
    fig.savefig(pdf_file, bbox_inches='tight', facecolor='white')
    
    print(f"\n✓ Saved PNG: {png_file}")
    print(f"✓ Saved PDF: {pdf_file}")
    
    plt.show()
    
    # ========================================================================
    # 打印摘要
    # ========================================================================
    
    print("\n" + "=" * 80)
    print("Figure 4 综合版摘要")
    print("=" * 80)
    print(f"\nEdge vs Interior (n={n_valid} folds):")
    print(f"  Top-1 accuracy:  Edge={med_acc_edge:.3f}, Interior={med_acc_int:.3f}")
    print(f"  Top-3 union:     Edge={med_top3_edge:.3f}, Interior={med_top3_int:.3f}")
    print(f"  Soft-ECE:        Edge={med_ece_edge:.4f}, Interior={med_ece_int:.4f}")
    
    print("\n✓ Figure 4 综合版生成完成！")

---

## Figure 4 综合版：2×3 布局

**第一行**：Edge vs Interior 对比（violin plots）
- (a) Top-1 accuracy - Edge vs Interior
- (b) Top-3 union - Edge vs Interior  
- (c) Soft-ECE - Edge vs Interior

**第二行**：Gross accuracy（整体，不分区域）
- (d) Top-1 gross accuracy - 所有 folds 的整体准确率
- (e) Top-3 gross accuracy - 所有 folds 的整体覆盖率
- (f) Sample info - 样本统计信息

## 使用说明

此代码实现了 **ISMRM Figure 4** 的完整分析流程。

### 核心功能

**三个指标对比（Edge vs Interior）**：
1. **Top-3 union (post-T)** - 温度缩放后的 Top-3 覆盖率
2. **Soft-ECE (post-T)** - 基于软参考的期望校准误差
3. **Top-1 accuracy (NOT post-T)** - Top-1 准确率（argmax 不受温度缩放影响）

### 关键特性

✅ **模块化设计** - 所有核心函数独立可测试  
✅ **内存安全** - NumPy-first，避免不必要的数据复制  
✅ **鲁棒性** - 自动处理 2D/3D 和 4D 输入  
✅ **6-connectivity 边界检测** - 符合医学图像标准  
✅ **自动保存** - PNG (300 dpi) + CSV 格式  

### 输出文件

- `figure4_edge_interior.png` - 三面板对比图
- `figure4_edge_interior_metrics.csv` - 数值结果

---

**注意**：此代码使用单个被试的数据作为示例。如需处理多个 folds，请修改数据加载部分。

In [ ]:
## Figure 4: Edge vs Interior - Production Code
# ============================================================================
# Modular, memory-safe, NumPy-first implementation
# ============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import distance_transform_edt
from pathlib import Path

# ============================================================================
# Core Functions
# ============================================================================

def compute_boundary_mask(y_int):
    """
    Compute boundary mask using 6-connectivity (3D) or 4-connectivity (2D).
    
    Parameters:
    - y_int: int32 array of shape [Z, Y, X] or [Y, X]
    
    Returns:
    - boundary: boolean array, True at boundary voxels
    """
    foreground = y_int > 0
    boundary = np.zeros_like(y_int, dtype=bool)
    
    if y_int.ndim == 3:
        # 3D: 6-connectivity (face neighbors only)
        for dz, dy, dx in [(-1,0,0), (1,0,0), (0,-1,0), (0,1,0), (0,0,-1), (0,0,1)]:
            shifted = np.roll(y_int, shift=(dz, dy, dx), axis=(0, 1, 2))
            boundary |= (y_int != shifted) & foreground
    elif y_int.ndim == 2:
        # 2D: 4-connectivity
        for dy, dx in [(-1,0), (1,0), (0,-1), (0,1)]:
            shifted = np.roll(y_int, shift=(dy, dx), axis=(0, 1))
            boundary |= (y_int != shifted) & foreground
    else:
        raise ValueError(f"y_int must be 2D or 3D, got shape {y_int.shape}")
    
    return boundary


def distance_to_boundary_mm(y_int, voxel_spacing_mm):
    """
    Compute Euclidean distance to nearest boundary in mm.
    
    Parameters:
    - y_int: int32 array [Z, Y, X] or [Y, X]
    - voxel_spacing_mm: tuple (sz, sy, sx) or (sy, sx) in mm
    
    Returns:
    - dist_mm: float array of same shape as y_int
    """
    boundary = compute_boundary_mask(y_int)
    dist_mm = distance_transform_edt(~boundary, sampling=voxel_spacing_mm)
    return dist_mm


def make_edge_interior_masks(y_int, voxel_spacing_mm, d_edge_mm=2.0, d_interior_mm=3.0):
    """
    Create Edge and Interior masks based on distance to boundary.
    
    Parameters:
    - y_int: int32 array [Z, Y, X] or [Y, X]
    - voxel_spacing_mm: tuple of voxel sizes
    - d_edge_mm: Edge threshold (≤ this distance)
    - d_interior_mm: Interior threshold (≥ this distance)
    
    Returns:
    - mask_edge: boolean array, flattened to [N]
    - mask_interior: boolean array, flattened to [N]
    """
    dist_mm = distance_to_boundary_mm(y_int, voxel_spacing_mm)
    foreground = y_int > 0
    
    mask_edge = (dist_mm <= d_edge_mm) & foreground
    mask_interior = (dist_mm >= d_interior_mm) & foreground
    
    # Flatten
    return mask_edge.ravel(), mask_interior.ravel()


def soft_ece_postT(P_postT, Q_soft, mask, n_bins=15):
    """
    Compute soft Expected Calibration Error (post-T).
    
    Parameters:
    - P_postT: [N, C] post-temperature posteriors
    - Q_soft: [N, C] soft reference probabilities
    - mask: [N] boolean mask
    - n_bins: number of confidence bins
    
    Returns:
    - ece: scalar soft-ECE value
    """
    # Apply mask
    P = P_postT[mask]
    Q = Q_soft[mask]
    
    if P.shape[0] == 0:
        return np.nan
    
    # Confidence = max(P, axis=-1)
    conf = P.max(axis=1)
    
    # Soft accuracy = sum(P * Q, axis=-1) (dot product)
    soft_acc = (P * Q).sum(axis=1)
    
    # Bin by confidence
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    
    for i in range(n_bins):
        if i == n_bins - 1:
            bin_mask = (conf >= bin_edges[i]) & (conf <= bin_edges[i + 1])
        else:
            bin_mask = (conf >= bin_edges[i]) & (conf < bin_edges[i + 1])
        
        n_b = bin_mask.sum()
        if n_b > 0:
            mean_conf = conf[bin_mask].mean()
            mean_soft_acc = soft_acc[bin_mask].mean()
            ece += np.abs(mean_conf - mean_soft_acc) * (n_b / P.shape[0])
    
    return ece


def top3_union_postT(P_postT, y_int, mask):
    """
    Compute Top-3 union coverage (post-T).
    
    Parameters:
    - P_postT: [N, C] post-temperature posteriors
    - y_int: [N] hard GT labels
    - mask: [N] boolean mask
    
    Returns:
    - coverage: scalar, fraction where y_int is in top-3
    """
    P = P_postT[mask]
    y = y_int[mask]
    
    if P.shape[0] == 0:
        return np.nan
    
    # Get top-3 indices for each sample
    top3_indices = np.argsort(P, axis=1)[:, -3:]  # [N, 3]
    
    # Check if y is in top-3
    matches = np.any(top3_indices == y[:, np.newaxis], axis=1)
    coverage = matches.mean()
    
    return coverage


def top1_accuracy(P_postT, y_int, mask):
    """
    Compute Top-1 accuracy (NOT post-T - argmax unchanged by temperature).
    
    Parameters:
    - P_postT: [N, C] posteriors (argmax same as pre-T)
    - y_int: [N] hard GT labels
    - mask: [N] boolean mask
    
    Returns:
    - accuracy: scalar, fraction of correct predictions
    """
    P = P_postT[mask]
    y = y_int[mask]
    
    if P.shape[0] == 0:
        return np.nan
    
    pred = P.argmax(axis=1)
    accuracy = (pred == y).mean()
    
    return accuracy


def figure4_edge_vs_interior(P_postT, Q_soft, y_int, voxel_spacing_mm):
    """
    Main function: compute all metrics and create Figure 4.
    
    Parameters:
    - P_postT: [Z, Y, X, C] or [N, C] post-T posteriors
    - Q_soft: [Z, Y, X, C] or [N, C] soft reference
    - y_int: [Z, Y, X] or [N] hard GT labels
    - voxel_spacing_mm: tuple (sz, sy, sx) or (sy, sx)
    
    Returns:
    - df: pandas DataFrame with metrics
    """
    # ========================================================================
    # Input validation and reshaping
    # ========================================================================
    
    # Handle 4D inputs
    if P_postT.ndim == 4:
        Z, Y, X, C = P_postT.shape
        P_postT_flat = P_postT.reshape(-1, C)
        Q_soft_flat = Q_soft.reshape(-1, C)
        y_int_spatial = y_int  # Keep for boundary computation
        y_int_flat = y_int.ravel()
    else:
        # Already 2D
        P_postT_flat = P_postT
        Q_soft_flat = Q_soft
        y_int_flat = y_int
        # Assume square-ish spatial layout for boundary (fallback)
        N, C = P_postT.shape
        side = int(np.sqrt(N))
        y_int_spatial = y_int.reshape(side, side)
    
    # Assertions
    assert P_postT_flat.shape == Q_soft_flat.shape, \
        f"Shape mismatch: P={P_postT_flat.shape}, Q={Q_soft_flat.shape}"
    assert P_postT_flat.shape[0] == y_int_flat.shape[0], \
        f"Sample count mismatch: P={P_postT_flat.shape[0]}, y={y_int_flat.shape[0]}"
    
    C = P_postT_flat.shape[1]
    
    # Filter valid labels [0, C-1]
    valid_labels = (y_int_flat >= 0) & (y_int_flat < C)
    P_postT_flat = P_postT_flat[valid_labels]
    Q_soft_flat = Q_soft_flat[valid_labels]
    y_int_flat = y_int_flat[valid_labels]
    
    print(f"Valid samples: {P_postT_flat.shape[0]} (after filtering labels)")
    
    # ========================================================================
    # Compute Edge and Interior masks
    # ========================================================================
    
    mask_edge_spatial, mask_interior_spatial = make_edge_interior_masks(
        y_int_spatial, voxel_spacing_mm, d_edge_mm=2.0, d_interior_mm=3.0
    )
    
    # Apply valid_labels mask
    mask_edge = mask_edge_spatial[valid_labels]
    mask_interior = mask_interior_spatial[valid_labels]
    
    N_edge = mask_edge.sum()
    N_interior = mask_interior.sum()
    
    print(f"\nRegion sizes:")
    print(f"  Edge (≤2mm):     {N_edge} voxels")
    print(f"  Interior (≥3mm): {N_interior} voxels")
    
    if N_edge < 100:
        print("  ⚠ Warning: Edge region has < 100 voxels!")
    if N_interior < 100:
        print("  ⚠ Warning: Interior region has < 100 voxels!")
    
    # ========================================================================
    # Compute metrics
    # ========================================================================
    
    print("\nComputing metrics...")
    
    # Edge metrics
    top3_edge = top3_union_postT(P_postT_flat, y_int_flat, mask_edge)
    ece_edge = soft_ece_postT(P_postT_flat, Q_soft_flat, mask_edge, n_bins=15)
    acc_edge = top1_accuracy(P_postT_flat, y_int_flat, mask_edge)
    
    # Interior metrics
    top3_interior = top3_union_postT(P_postT_flat, y_int_flat, mask_interior)
    ece_interior = soft_ece_postT(P_postT_flat, Q_soft_flat, mask_interior, n_bins=15)
    acc_interior = top1_accuracy(P_postT_flat, y_int_flat, mask_interior)
    
    # Print summary
    print("\nMetrics Summary:")
    print(f"  Top-3 union (post-T):  Edge={top3_edge:.4f}, Interior={top3_interior:.4f}")
    print(f"  Soft-ECE (post-T):     Edge={ece_edge:.4f}, Interior={ece_interior:.4f}")
    print(f"  Top-1 accuracy:        Edge={acc_edge:.4f}, Interior={acc_interior:.4f}")
    
    # ========================================================================
    # Create DataFrame
    # ========================================================================
    
    df = pd.DataFrame([
        {"region": "Edge (≤2mm)", "top3_union_postT": top3_edge, 
         "soft_ece_postT": ece_edge, "top1_accuracy": acc_edge},
        {"region": "Interior (≥3mm)", "top3_union_postT": top3_interior, 
         "soft_ece_postT": ece_interior, "top1_accuracy": acc_interior}
    ])
    
    # ========================================================================
    # Plotting
    # ========================================================================
    
    print("\nGenerating Figure 4...")
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.patch.set_facecolor('white')
    
    x_labels = ["Edge\n(≤2 mm)", "Interior\n(≥3 mm)"]
    x_pos = [0, 1]
    
    # (a) Top-3 union (post-T)
    values_top3 = [top3_edge, top3_interior]
    bars_a = axes[0].bar(x_pos, values_top3, width=0.6)
    axes[0].set_ylabel('Coverage', fontsize=11)
    axes[0].set_title('(a) Top-3 union (post-T)', fontsize=12, fontweight='bold')
    axes[0].set_xticks(x_pos)
    axes[0].set_xticklabels(x_labels, fontsize=10)
    axes[0].set_ylim([0, 1])
    axes[0].grid(axis='y', alpha=0.3, linestyle='--')
    
    # Add value labels
    for i, (bar, val) in enumerate(zip(bars_a, values_top3)):
        if not np.isnan(val):
            axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.02, 
                        f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    # (b) Soft-ECE (post-T)
    values_ece = [ece_edge, ece_interior]
    bars_b = axes[1].bar(x_pos, values_ece, width=0.6)
    axes[1].set_ylabel('ECE', fontsize=11)
    axes[1].set_title('(b) soft-ECE (post-T, soft reference)', fontsize=12, fontweight='bold')
    axes[1].set_xticks(x_pos)
    axes[1].set_xticklabels(x_labels, fontsize=10)
    axes[1].set_ylim([0, max(values_ece) * 1.2 if not all(np.isnan(values_ece)) else 0.1])
    axes[1].grid(axis='y', alpha=0.3, linestyle='--')
    
    for i, (bar, val) in enumerate(zip(bars_b, values_ece)):
        if not np.isnan(val):
            axes[1].text(bar.get_x() + bar.get_width()/2, val + max(values_ece)*0.02, 
                        f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    # (c) Top-1 accuracy (NOT post-T)
    values_acc = [acc_edge, acc_interior]
    bars_c = axes[2].bar(x_pos, values_acc, width=0.6)
    axes[2].set_ylabel('Accuracy', fontsize=11)
    axes[2].set_title('(c) Top-1 accuracy (not post-T)', fontsize=12, fontweight='bold')
    axes[2].set_xticks(x_pos)
    axes[2].set_xticklabels(x_labels, fontsize=10)
    axes[2].set_ylim([0, 1])
    axes[2].grid(axis='y', alpha=0.3, linestyle='--')
    
    for i, (bar, val) in enumerate(zip(bars_c, values_acc)):
        if not np.isnan(val):
            axes[2].text(bar.get_x() + bar.get_width()/2, val + 0.02, 
                        f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    # Suptitle
    fig.suptitle('Figure 4 — Interface vs interior (post-T calibration & correctness)',
                 fontsize=14, fontweight='bold', y=0.98)
    
    # Footnote
    footnote = ("Edge: dist≤2 mm; Interior: dist≥3 mm; confidence bins=15; "
                "temperature-scaled metrics reported post-T only.")
    fig.text(0.5, 0.02, footnote, ha='center', fontsize=8, style='italic', color='gray')
    
    plt.tight_layout(rect=[0, 0.04, 1, 0.96])
    
    # ========================================================================
    # Save outputs
    # ========================================================================
    
    # Save figure
    output_dir = Path(OUTPUT_DIR) if 'OUTPUT_DIR' in globals() else Path("./output")
    output_dir.mkdir(parents=True, exist_ok=True)
    
    png_file = output_dir / "figure4_edge_interior.png"
    csv_file = output_dir / "figure4_edge_interior_metrics.csv"
    
    fig.savefig(png_file, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"\n✓ Saved PNG: {png_file}")
    
    df.to_csv(csv_file, index=False, float_format='%.6f')
    print(f"✓ Saved CSV: {csv_file}")
    
    plt.show()
    
    return df


# ============================================================================
# Main execution
# ============================================================================

print("=" * 80)
print("Figure 4: Edge vs Interior Analysis (Production Version)")
print("=" * 80)

# Load data (adapt to your session variables)
print("\nLoading data...")

# Example: Load from existing variables or files
# Assuming data already exists in session from previous cells
try:
    # From previous Figure 4 code
    pred_data = np.load(PRED_SOFTMAX_FILE)
    gt_data = np.load(GT_3D_FILE)
    
    P_postT = pred_data['pred_softmax_3d']  # [Z, Y, X, C]
    Q_soft = gt_data['proba_labels']  # [Z, Y, X, C]
    region_mask = gt_data['region_mask_lr']  # [Z, Y, X]
    
    # Create hard labels from soft reference
    y_int = np.argmax(Q_soft, axis=-1).astype(np.int32)  # [Z, Y, X]
    
    # Apply region mask (set background to -1 to filter later)
    y_int[region_mask == 0] = -1
    
    voxel_spacing_mm = (3.0, 1.8, 1.8)  # (Z, Y, X) in mm
    
    print(f"  P_postT shape: {P_postT.shape}")
    print(f"  Q_soft shape: {Q_soft.shape}")
    print(f"  y_int shape: {y_int.shape}")
    print(f"  Voxel spacing: {voxel_spacing_mm} mm")
    
    # Run analysis
    df_metrics = figure4_edge_vs_interior(P_postT, Q_soft, y_int, voxel_spacing_mm)
    
    print("\n" + "=" * 80)
    print("✓ Figure 4 completed successfully!")
    print("=" * 80)
    print("\nMetrics DataFrame:")
    print(df_metrics.to_string(index=False))
    
except Exception as e:
    print(f"\n✗ Error: {e}")
    import traceback
    traceback.print_exc()

---

# Figure 4: Edge vs Interior Analysis (Production Version)

**Three metrics comparison**:
1. Top-3 union (post-T)
2. Soft-ECE (post-T, soft reference)
3. Top-1 accuracy (NOT post-T)

**Requirements**: NumPy-first, matplotlib only, modular functions, memory-safe